## Задача


\begin{cases}
\dfrac{dx}{dt} = x \left( 1 - 0.5x - \dfrac{2}{7} \alpha_2^{-2} y \right), \quad x(0) = x_0, \\[10pt]
\dfrac{dy}{dt} = y \left( 2\alpha_2 - 0.5y - 3.5\alpha_2^2 x \right), \quad y(0) = y_0, \\[10pt]
\dfrac{d\alpha_2}{dt} = \varepsilon (2 - 7\alpha_2 x), \quad \alpha_2(0) = \alpha_{20}; \quad t \in [0; T_k].
\end{cases}

#### Рекомендуемые значения начальных данных


$0 \le x_0 \le 3$,

 $0 \le y_0 \le 15$
 
 $\alpha_{20}$ близко к нулю, например,
можно положить $\alpha_{20} = 0.0001$;

 $T_k = 1500$. 
 
$\varepsilon \le 0.01$ 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

In [ ]:
def system_func(t, v, epsilon=0.01):
    x, y, a2 = v
    a2_reg = a2 if abs(a2) > 1e-10 else 1e-10
    
    dxdt = x * (1 - 0.5 * x - (2/7) * (a2_reg**-2) * y)
    dydt = y * (2 * a2 - 0.5 * y - 3.5 * (a2_reg**2) * x)
    da2dt = epsilon * (2 - 7 * a2 * x)
    
    return np.array([dxdt, dydt, da2dt])

In [ ]:
def rk4_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h/2, y + h/2 * k1)
    k3 = f(t + h/2, y + h/2 * k2)
    k4 = f(t + h, y + h * k3)
    return y + h/6 * (k1 + 2*k2 + 2*k3 + k4)

def adams_bashfort(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    steps = len(t) - 1 
    mas = [y0]
    for i in range(3):
        mas.append(rk4_step(f, t[i], mas[i], h))
    vec = []
    vec.append(f(t[0], mas[0]))
    vec.append(f(t[1], mas[1]))
    vec.append(f(t[2], mas[2]))
    
    for j in range(3, steps):
        vec.append(f(t[j], mas[j]))
        f_0 = vec[j]
        f_1 = vec[j-1]
        f_2 = vec[j-2]
        f_3 = vec[j-3]
        y_next = mas[j] + (h / 24.0) * (55*f_0 - 59*f_1 + 37*f_2 - 9*f_3)
        mas.append(y_next)
    return t, np.array(mas)


def adams_moulton(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    steps = len(t) - 1
    mas = [y0]
    for i in range(3):
        mas.append(rk4_step(f, t[i], mas[i], h))
        
    vec = [f(t[i], mas[i]) for i in range(3)]
    max_iters = 0
    iters_sum = 0
    count = 0

    for j in range(3, steps):
        vec.append(f(t[j], mas[j]))
        f_0, f_1, f_2 = vec[j], vec[j-1], vec[j-2]

        y1 = mas[j]
        # Будущая точка t[j+1] передается в функцию f
        y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
        
        iters = 1
        while np.max(np.abs(y2 - y1)) > 1e-12:
            y1 = y2
            y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
            iters += 1
            if np.any(np.isnan(y2)) or np.any(np.isinf(y2)):
                break
        
        mas.append(y2)
        iters_sum += iters
        max_iters = max(max_iters, iters)
        count += 1
        
    avg_iters = iters_sum / count if count > 0 else -1
    print(f"Adams-Moulton [h={h}]: Max Iters = {max_iters}, Avg Iters = {avg_iters:.2f}")
    
    return t, np.array(mas)


def adams_bash_moulton(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    steps = len(t) - 1
    
    mas = [y0]
    for i in range(3):
        mas.append(rk4_step(f, t[i], mas[i], h))
        
    vec = [f(t[i], mas[i]) for i in range(3)]
    max_iters = 0
    iters_sum = 0
    count = 0

    for j in range(3, steps):
        vec.append(f(t[j], mas[j]))
        f_0, f_1, f_2, f_3 = vec[j], vec[j-1], vec[j-2], vec[j-3]

        y1 = mas[j] + (h / 24.0) * (55*f_0 - 59*f_1 + 37*f_2 - 9*f_3)
        y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
        
        iters = 1
        while np.max(np.abs(y2 - y1)) > 1e-12:
            y1 = y2
            y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
            iters += 1
            if np.any(np.isnan(y2)) or np.any(np.isinf(y2)):
                break
                
        mas.append(y2)
        iters_sum += iters
        max_iters = max(max_iters, iters)
        count += 1
        
    avg_iters = iters_sum / count if count > 0 else -1
    print(f"Bashforth-Moulton (Предиктор-Корректор) [h={h}]: Max Iters = {max_iters}, Avg Iters = {avg_iters:.2f}")
    
    return t, np.array(mas)

In [ ]:
import pandas as pd

def compute_convergence_orders(methods, system_func, y0, t_span, h_start, n_refinements, ethalon_sol=None):
    data = []
    prev_results = {name: None for name in methods}
    
    h = h_start
    for i in range(n_refinements):
        row = {'h': h}
        t_eval = np.arange(t_span[0], t_span[1] + h, h)
        
        for name, method_func in methods.items():
            # ТЕПЕРЬ ЛОВИМ 4 ЗНАЧЕНИЯ, включая итерации!
            t_curr, y_curr, max_iter, avg_iter = method_func(system_func, y0, t_span, h)
            
            # Считаем ошибку
            if ethalon_sol is not None:
                y_true = ethalon_sol(t_curr)
                error = np.max(np.abs(y_curr - y_true))
            else:
                if prev_results[name] is not None:
                    error = np.max(np.abs(y_curr[::2] - prev_results[name]))
                else:
                    error = np.nan
            
            # Записываем ошибку в строку
            row[name] = error
            
            # ЗАПИСЫВАЕМ ИТЕРАЦИИ В СТРОКУ (только для неявных методов)
            if "Bashforth 4" not in name:  # Явный метод пропускаем, чтобы не мусорить нулями
                row[f'Iters_{name}'] = round(avg_iter, 2)
                
            prev_results[name] = y_curr
            
            # Считаем порядок сходимости (p)
            p_key = f"p_{name}"
            if i > 0 and not np.isnan(data[i-1][name]) and error > 0:
                order = np.log2(data[i-1][name] / error)
                row[p_key] = round(order, 2)
            else:
                row[p_key] = np.nan
                
        data.append(row)
        h /= 2  # Уменьшаем шаг в 2 раза для следующего круга
        
    return pd.DataFrame(data)

In [ ]:
y0_system = [1.5, 7.5, 0.01]
t_range = [0, 0.01]
eps_param = 0.01

pure_system = lambda t, v: system_func(t, v, eps_param)
sol = solve_ivp(pure_system, t_range, y0_system, method='BDF', atol=1e-12, rtol=1e-12)
ethalon_func = interp1d(sol.t, sol.y, axis=1, kind='cubic', fill_value="extrapolate")

my_methods = {
    "Adams-Bashforth 4": adams_bashfort,
    "Adams-Moulton 4": adams_moulton,
    "Adams-Bashforth-Moulton": adams_bash_moulton
}

df_results = compute_convergence_orders(
    my_methods, 
    pure_system, 
    y0_system, 
    t_range, 
    h_start=0.0001, 
    n_refinements=5, 
    ethalon_sol=lambda t: ethalon_func(t).T
)

df_results

In [ ]:
def analyze_system_stability(v, h, epsilon=0.01):
    """
    v: текущее состояние [x, y, a2]
    h: шаг интегрирования
    epsilon: параметр системы
    """
    x, y, a2 = v
    
    J = np.zeros((3, 3))
    
    J[0, 0] = 1 - x - (2/7) * (a2**-2) * y
    J[0, 1] = -(2/7) * (a2**-2) * x
    J[0, 2] = (4/7) * (a2**-3) * x * y
    
    J[1, 0] = -3.5 * (a2**2) * y
    J[1, 1] = 2 * a2 - y - 3.5 * (a2**2) * x
    J[1, 2] = 2 * y - 7 * a2 * x * y
    
    J[2, 0] = -7 * epsilon * a2
    J[2, 1] = 0
    J[2, 2] = -7 * epsilon * x
    
    lambdas = np.linalg.eigvals(J)
    
    print(f"--- Анализ устойчивости (h={h}) ---")
    print(f"Точка: x={x:.2f}, y={y:.2f}, a2={a2:.5f}")
    
    is_stable_overall = True
    for i, lam in enumerate(lambdas):
        z = h * lam
        stability_radius = np.abs(1 + z)
        
        status = "OK" if stability_radius <= 1 else "ВЗРЫВ (Неустойчиво)"
        if stability_radius > 1: is_stable_overall = False
        
        print(f"lambda_{i+1} = {lam:.2e} | |1 + h*lambda| = {stability_radius:.4f} -> {status}")
    
    if not is_stable_overall:
        max_h = 2.0 / np.max(np.abs(lambdas))
        print(f"СОВЕТ: Для устойчивости в этой точке шаг h должен быть меньше {max_h:.2e}")
    
    return J, lambdas

# Пример использования для твоих начальных данных
y0_initial = [1.5, 7.5, 0.0001]
analyze_system_stability(y0_initial, h=0.1)
analyze_system_stability(y0_initial, h=0.1)
y0_initial = [1.5, 7.5, 0.0001]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. ГРАФИК НАЛОЖЕНИЯ НА ИДЕАЛЬНОЕ РЕШЕНИЕ ---
# Берем небольшой отрезок времени, где методы начинают расходиться
t_range_plot = [0, 0.0001]
h_value = 0.000013 # Берем шаг, на котором Башфорт еще жив, но уже ошибается

# Вычисляем методы
t_ab, y_ab = adams_bashfort(pure_system, y0_initial, t_range_plot, h=h_value)
t_am, y_am = adams_moulton(pure_system, y0_initial, t_range_plot, h=h_value)
t_abm, y_abm = adams_bash_moulton(pure_system, y0_initial, t_range_plot, h=h_value)

# Вычисляем идеальное решение на тех же точках t
ideal_y = ethalon_func(t_ab).T 

plt.figure(figsize=(12, 7))
# Рисуем идеальное решение толстой черной линией
plt.plot(t_ab, ideal_y[:, 0], 'k-', label='Идеальное решение x(t)', linewidth=3)

# Накладываем наши методы
plt.plot(t_ab, y_ab[:, 0], 'r--', label='Adams-Bashforth (Явный)', linewidth=2)
plt.plot(t_am, y_am[:, 0], 'b-.', label='Adams-Moulton (Неявный)', linewidth=2)
plt.plot(t_abm, y_abm[:, 0], 'g:', label='Predictor-Corrector', linewidth=2)

plt.xlabel('Время (t)', fontsize=12)
plt.ylabel('Значение x', fontsize=12)
plt.title(f'Наложение методов на идеальное решение (при шаге h={h_value})', fontsize=14)
plt.legend()
plt.grid(True)
plt.show()


# --- 2. ГРАФИК УБЫВАНИЯ ОШИБКИ (СХОДИМОСТЬ) ---
# Используем твой датафрейм df_results, который есть на скриншоте
plt.figure(figsize=(10, 6))

# Рисуем линии для каждого метода (h vs Ошибка)
methods_to_plot = ["Adams-Bashforth 4", "Adams-Moulton 4", "Adams-Bashforth-Moulton"]
colors = ['red', 'blue', 'green']

for name, col in zip(methods_to_plot, colors):
    if name in df_results.columns:
        valid_mask = ~df_results[name].isna()
        plt.loglog(df_results.loc[valid_mask, 'h'], df_results.loc[valid_mask, name], 
                   marker='o', linestyle='-', color=col, label=name, linewidth=2)

plt.xlabel('Шаг интегрирования (h)', fontsize=12)
plt.ylabel('Максимальная абсолютная ошибка (Log Scale)', fontsize=12)
plt.title('График убывания ошибки методов Адамса', fontsize=14)
# Инвертируем ось X, чтобы шаг h уменьшался слева направо
plt.gca().invert_xaxis() 
plt.grid(True, which="both", ls="--", alpha=0.7)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

# --- 1. ЯВНЫЙ МЕТОД АДАМСА-БАШФОРТА ---
def adams_bashfort(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    steps = len(t) - 1
    
    mas = [y0]
    for i in range(3):
        mas.append(rk4_step(f, t[i], mas[i], h))
        
    vec = [f(t[i], mas[i]) for i in range(3)]
    
    for j in range(3, steps):
        vec.append(f(t[j], mas[j]))
        f_0, f_1, f_2, f_3 = vec[j], vec[j-1], vec[j-2], vec[j-3]
        y_next = mas[j] + (h / 24.0) * (55*f_0 - 59*f_1 + 37*f_2 - 9*f_3)
        mas.append(y_next)
        
    # Явный метод: итераций нет, отдаем нули
    return t, np.array(mas), 0, 0.0 


# --- 2. НЕЯВНЫЙ МЕТОД АДАМСА-МОУЛТОНА ---
def adams_moulton(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    steps = len(t) - 1
    
    mas = [y0]
    for i in range(3):
        mas.append(rk4_step(f, t[i], mas[i], h))
        
    vec = [f(t[i], mas[i]) for i in range(3)]
    max_iters = 0
    iters_sum = 0
    count = 0

    for j in range(3, steps):
        vec.append(f(t[j], mas[j]))
        f_0, f_1, f_2 = vec[j], vec[j-1], vec[j-2]

        y1 = mas[j]
        y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
        
        iters = 1
        # ПРЕДОХРАНИТЕЛЬ: iters < 100, чтобы не зависало
        while np.max(np.abs(y2 - y1)) > 1e-12 and iters < 100:
            y1 = y2
            y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
            iters += 1
            if np.any(np.isnan(y2)) or np.any(np.isinf(y2)):
                break
        
        mas.append(y2)
        iters_sum += iters
        max_iters = max(max_iters, iters)
        count += 1
        
    avg_iters = iters_sum / count if count > 0 else 0.0
    return t, np.array(mas), max_iters, avg_iters


# --- 3. АДАМС-БАШФОРТ-МОУЛТОН ---
def adams_bash_moulton(f, y0, t_span, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    steps = len(t) - 1
    
    mas = [y0]
    for i in range(3):
        mas.append(rk4_step(f, t[i], mas[i], h))
        
    vec = [f(t[i], mas[i]) for i in range(3)]
    max_iters = 0
    iters_sum = 0
    count = 0

    for j in range(3, steps):
        vec.append(f(t[j], mas[j]))
        f_0, f_1, f_2, f_3 = vec[j], vec[j-1], vec[j-2], vec[j-3]

        y1 = mas[j] + (h / 24.0) * (55*f_0 - 59*f_1 + 37*f_2 - 9*f_3)
        y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
        
        iters = 1
        # ПРЕДОХРАНИТЕЛЬ: iters < 100
        while np.max(np.abs(y2 - y1)) > 1e-12 and iters < 100:
            y1 = y2
            y2 = mas[j] + (h / 24.0) * (9*f(t[j+1], y1) + 19*f_0 - 5*f_1 + f_2)
            iters += 1
            if np.any(np.isnan(y2)) or np.any(np.isinf(y2)):
                break
                
        mas.append(y2)
        iters_sum += iters
        max_iters = max(max_iters, iters)
        count += 1
        
    avg_iters = iters_sum / count if count > 0 else 0.0
    return t, np.array(mas), max_iters, avg_iters


# --- 4. ФУНКЦИЯ ДЛЯ СБОРКИ ТАБЛИЦЫ ---
def compute_convergence_orders(methods, system_func, y0, t_span, h_start, n_refinements, ethalon_sol=None):
    data = []
    prev_results = {name: None for name in methods}
    
    h = h_start
    for i in range(n_refinements):
        row = {'h': h}
        t_eval = np.arange(t_span[0], t_span[1] + h, h)
        
        for name, method_func in methods.items():
            # ЛОВИМ ВСЕ 4 ЗНАЧЕНИЯ ИЗ МЕТОДОВ
            t_curr, y_curr, max_iter, avg_iter = method_func(system_func, y0, t_span, h)
            
            if ethalon_sol is not None:
                y_true = ethalon_sol(t_curr)
                error = np.max(np.abs(y_curr - y_true))
            else:
                if prev_results[name] is not None:
                    error = np.max(np.abs(y_curr[::2] - prev_results[name]))
                else:
                    error = np.nan
            
            row[name] = error
            
            # Если у метода есть итерации (он неявный), добавляем их в таблицу
            if avg_iter > 0:
                row[f'Iters_{name}'] = round(avg_iter, 2)
                
            prev_results[name] = y_curr
            
            p_key = f"p_{name}"
            if i > 0 and not pd.isna(data[i-1][name]) and error > 0:
                order = np.log2(data[i-1][name] / error)
                row[p_key] = round(order, 2)
            else:
                row[p_key] = np.nan
                
        data.append(row)
        h /= 2
        
    return pd.DataFrame(data)

# --- 5. ЗАПУСК ИССЛЕДОВАНИЯ ---
y0_system = [1.5, 7.5, 0.01]
t_range = [0, 0.01]
eps_param = 0.01

pure_system = lambda t, v: system_func(t, v, eps_param)
sol = solve_ivp(pure_system, t_range, y0_system, method='BDF', atol=1e-12, rtol=1e-12)
ethalon_func = interp1d(sol.t, sol.y, axis=1, kind='cubic', fill_value="extrapolate")

my_methods = {
    "Adams-Bashforth 4": adams_bashfort,
    "Adams-Moulton 4": adams_moulton,
    "Adams-Bashforth-Moulton 4": adams_bash_moulton
}

df_results = compute_convergence_orders(
    my_methods, 
    pure_system, 
    y0_system, 
    t_range, 
    h_start=0.0001, # Начальный шаг
    n_refinements=5, 
    ethalon_sol=lambda t: ethalon_func(t).T
)

df_results

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Названия колонок в точности как в твоей таблице
methods_to_plot = ["Adams-Bashforth 4", "Adams-Moulton 4", "Adams-Bashforth-Moulton 4"]
colors = ['red', 'blue', 'green']
markers = ['o', 's', '^']

for name, col, mark in zip(methods_to_plot, colors, markers):
    if name in df_results.columns:
        # Отфильтруем NaN значения (где Башфорт взорвался), чтобы график построился
        valid_mask = ~df_results[name].isna()
        plt.loglog(df_results.loc[valid_mask, 'h'], df_results.loc[valid_mask, name], 
                   marker=mark, linestyle='-', color=col, label=name, linewidth=2, markersize=8)

plt.xlabel('Шаг интегрирования $h$', fontsize=12)
plt.ylabel('Максимальная абсолютная ошибка', fontsize=12)
plt.title('Сходимость методов Адамса (Логарифмический масштаб)', fontsize=14)
# Инвертируем ось X, так как шаг h у нас уменьшается слева направо
plt.gca().invert_xaxis() 
plt.grid(True, which="both", ls="--", alpha=0.7)
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Берем шаг из таблицы, где Башфорт дает видимую ошибку (0.000744)
h_test = 0.000013 
t_span_test = [0, 0.0001]

# Запускаем расчеты
t_ab, y_ab, _, _ = adams_bashfort(pure_system, y0_system, t_span_test, h=h_test)
t_am, y_am, _, _ = adams_moulton(pure_system, y0_system, t_span_test, h=h_test)
t_abm, y_abm, _, _ = adams_bash_moulton(pure_system, y0_system, t_span_test, h=h_test)

# Получаем эталонное (идеальное) решение для этих же точек времени
ideal_y = ethalon_func(t_ab).T 

plt.figure(figsize=(12, 7))
# Идеальное решение рисуем толстой полупрозрачной линией на фоне
plt.plot(t_ab, ideal_y[:, 0], 'k-', label='Идеальное решение x(t)', linewidth=5, alpha=0.3)

# Накладываем наши методы
plt.plot(t_ab, y_ab[:, 0], 'r--', label='Adams-Bashforth 4 (Явный)', linewidth=2)
plt.plot(t_am, y_am[:, 0], 'b-.', label='Adams-Moulton 4 (Неявный)', linewidth=2)
plt.plot(t_abm, y_abm[:, 0], 'g:', label='Adams-Bashforth-Moulton 4', linewidth=2)

plt.xlabel('Время (t)', fontsize=12)
plt.ylabel('Численность популяции x(t)', fontsize=12)
plt.title(f'Динамика сходимости к эталонному решению (при шаге h={h_test})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Задаем углы для отрисовки кривой
theta = np.linspace(0, 2*np.pi, 1000)
r = np.exp(1j * theta)

# 1. Полиномы для Adams-Bashforth 4 (явный)
rho_ab = r**4 - r**3
sigma_ab = (55*r**3 - 59*r**2 + 37*r - 9) / 24
hl_ab = rho_ab / sigma_ab

# 2. Полиномы для Adams-Moulton 4 (неявный)
# Так как АБМ и Моултон решают одно и то же уравнение до сходимости, их полиномы идентичны
rho_am = r**3 - r**2
sigma_am = (9*r**3 + 19*r**2 - 5*r + 1) / 24
hl_am = rho_am / sigma_am

plt.figure(figsize=(11, 7))

# Рисуем Башфорта (красный)
plt.plot(hl_ab.real, hl_ab.imag, 'r-', linewidth=2, label='Adams-Bashforth 4 (Явный)')

# Рисуем чистого Моултона (толстая синяя линия)
plt.plot(hl_am.real, hl_am.imag, 'b-', linewidth=5, alpha=0.4, label='Adams-Moulton 4 (Неявный)')

# Рисуем Предиктор-Корректор (зеленый пунктир поверх синей линии)
plt.plot(hl_am.real, hl_am.imag, 'g--', linewidth=2, label='Adams-Bashforth-Moulton 4 (Предиктор-Корректор)')

# Оси координат
plt.axhline(0, color='black', lw=1)
plt.axvline(0, color='black', lw=1)

plt.xlabel('Re(hλ)', fontsize=12)
plt.ylabel('Im(hλ)', fontsize=12)
plt.title('Области абсолютной устойчивости методов Адамса', fontsize=14)

# РАСШИРИЛИ ГРАНИЦЫ, чтобы увидеть, где заканчивается неявный метод
plt.xlim(-3.5, 0.5)
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.legend(loc='upper left', fontsize=11)

# Подсказка
plt.text(-3.4, -1.3, 
         "Явный метод (красный) ограничен Re(hλ) ≈ -0.3.\n", 
         fontsize=10, bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

x0 = 1.5           # Начальная популяция X (жертвы) [от 0 до 3]
y0 = 5.5           # Начальная популяция Y (хищники) [от 0 до 15]
a20 = 0.0001       # Начальная генетика (оставляем 0.0001)
eps_param = 0.01   # Скорость адаптации (пробуй 0.01, 0.005, 0.001)
T_k = 1500         # Время наблюдения

y0_eco = [x0, y0, a20]
t_range_eco = [0, T_k]

# Создаем функцию с текущим эпсилон
pure_system_eco = lambda t, v: system_func(t, v, eps_param)

# Считаем встроенным мощным методом BDF (он адаптивный и быстрый)
print(f"Запуск симуляции до T_k={T_k} с eps={eps_param}...")
sol_eco = solve_ivp(pure_system_eco, t_range_eco, y0_eco, method='BDF', 
                    dense_output=True, atol=1e-9, rtol=1e-9)

# Вытаскиваем результаты
t_eco = sol_eco.t
x_eco = sol_eco.y[0]
y_eco = sol_eco.y[1]
a2_eco = sol_eco.y[2]

# --- СТРОИМ КРАСИВЫЕ ЭКОЛОГИЕСКИЕ ГРАФИКИ ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Экогенетическая модель: x0={x0}, y0={y0}, eps={eps_param}', fontsize=14)

# График 1: Популяции со временем
ax1.plot(t_eco, x_eco, 'b-', label='Популяция X (Жертвы)', linewidth=2)
ax1.plot(t_eco, y_eco, 'r-', label='Популяция Y (Хищники)', linewidth=2)
ax1.set_xlabel('Время (t)')
ax1.set_ylabel('Численность')
ax1.set_title('Динамика численности')
ax1.grid(True)
ax1.legend()

# График 2: Фазовый портрет (Взаимодействие видов)
ax2.plot(x_eco, y_eco, 'g-', linewidth=1.5)
ax2.plot(x_eco[0], y_eco[0], 'ko', markersize=8, label='Старт') # Точка старта
ax2.plot(x_eco[-1], y_eco[-1], 'ro', markersize=8, label='Финиш') # Точка финиша
ax2.set_xlabel('Численность X')
ax2.set_ylabel('Численность Y')
ax2.set_title('Фазовый портрет (X от Y)')
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()

# График 3: Эволюция генетического признака
plt.figure(figsize=(7, 4))
plt.plot(t_eco, a2_eco, 'm-', linewidth=2, label='Признак alpha_2')
plt.xlabel('Время (t)')
plt.ylabel('Значение alpha_2')
plt.title('Эволюция генетического признака')
plt.grid(True)
plt.legend()
plt.show()